In [19]:
# обязательно запустить, чтобы загружать модули через model.
import sys
from pathlib import Path
if str(Path().resolve().parent) not in sys.path:
    sys.path.insert(0, str(Path().resolve().parent))

PROJECT_PATH = Path.cwd().parent
GENERAL_PATH = PROJECT_PATH / "coco_yolo"
TEST_PATH =    PROJECT_PATH / "dataset/test"
WEIGHTS_PATH = PROJECT_PATH / "model/runs/weights"

# Обучение

## Визуализация входных данных

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import torch
from data_proc import YOLODataset

def visualize_sample(dataset, idx=0, class_names=None):
    img, targets = dataset[idx]
    
    if isinstance(img, torch.Tensor):
        img = img.permute(1, 2, 0).cpu().numpy()
    img_vis = (img * 255).astype(np.uint8)

    if isinstance(targets, torch.Tensor):
        targets = targets.cpu().numpy()
    
    H, W = img_vis.shape[:2]
    
    fig, ax = plt.subplots(1, figsize=(12, 8))
    ax.imshow(img_vis)
    ax.axis('off')
    
    colors = plt.cm.tab10(np.linspace(0, 1, 10))
    
    for target in targets:
        if len(target) == 0:
            continue
        cls_id, x, y, w, h = target
        
        cx, cy = x * W, y * H
        bw, bh = w * W, h * H
        x1 = cx - bw / 2
        y1 = cy - bh / 2
        
        color = colors[int(cls_id) % len(colors)]
        
        # Рисуем прямоугольник
        rect = Rectangle(
            (x1, y1), bw, bh,
            linewidth=2, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        
        label = class_names[int(cls_id)] if class_names else f"cls{int(cls_id)}"
        ax.text(
            x1, y1 - 5, label,
            color='white', fontsize=10,
            bbox=dict(facecolor=color, alpha=0.7, edgecolor='none', pad=0.5)
        )
    
    plt.title(f"Sample {idx} | {len(targets)} objects")
    plt.tight_layout()
    plt.show()
    return fig

train_dataset = YOLODataset(GENERAL_PATH + "images/train", GENERAL_PATH + "labels/train", GENERAL_PATH + "train_patches", imgsz=640, augment=True)
class_names = {0: 'ball', 1: 'coach', 2: 'gk', 3: 'player', 4: 'ref'}
res = visualize_sample(train_dataset, idx=1, class_names=class_names)

## Функции обучения

In [ ]:
import torch
import torch.optim as optim
from tqdm import tqdm
import time
from collections import defaultdict

@torch.no_grad()
def class_weight(targets, num_classes, device):
    class_count = defaultdict(int)
    for t in targets:
        if t.numel() == 0:
            continue
        class_ids = t[:, 0].long().cpu().numpy()
        for cls in class_ids:
            class_count[cls] += 1
    total_samples = sum(class_count.values())
    
    cls_weight = []
    for cls in range(num_classes):
        pos_count = class_count.get(cls, 0)
        neg_count = total_samples - pos_count
        if pos_count == 0:
            weight = 1.0
        else:
            weight = neg_count / pos_count
        cls_weight.append(weight)

    max_alpha = 15.0
    pos_weight = torch.clamp(torch.tensor(cls_weight, device=device, requires_grad=False), max=max_alpha)
    
    return pos_weight

def train_one_epoch(model, dataloader, optimizer, scaler, loss_fn, device, epoch, imgsz=640):
    model.train()
    total_loss = 0.0
    loss_logs = {"loss_box": 0.0, "loss_obj": 0.0, "loss_cls": 0.0, "num_pos": 0}

    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"Epoch {epoch}")
    for batch_idx, (images, targets) in pbar:
        images = images.to(device)
        targets = [t.to(device) for t in targets]

        cls_weight = class_weight(targets, model.nc_external, device)
        gamma_per_class = torch.ones(model.nc_external, device=device)
        gamma_per_class[0] = 4.0
        gamma_per_class[1]= 4.5
        gamma_per_class[2] = 4.5

        with torch.amp.autocast(device.type):
            optimizer.zero_grad()
            pred = model(images)
        
            loss, logs = loss_fn(
                pred, targets,
                pos_weight=cls_weight,
                gamma_per_class=gamma_per_class,
                num_classes=model.nc_external,
                imgsz=imgsz,
                device=device,
                weight_box=7.5,   # CIoU/GIoU loss обычно сильнее
                weight_obj=10.0,
                weight_cls=0.5    # классы — менее критичны, чем bbox
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        for k in loss_logs:
            loss_logs[k] += logs[k]

        pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "obj": f"{logs['loss_obj']:.4f}",
            "cls": f"{logs['loss_cls']:.4f}",
            "box": f"{logs['loss_box']:.4f}",
            "pos": f"{logs['num_pos']}"
        })

    avg_loss = total_loss / len(dataloader)
    for k in loss_logs:
        loss_logs[k] /= len(dataloader)

    return avg_loss, loss_logs

@torch.no_grad()
def validate_one_epoch(model, dataloader, loss_fn, device, imgsz=640):
    model.eval()
    total_loss = 0.0
    loss_logs = {"loss_box": 0.0, "loss_obj": 0.0, "loss_cls": 0.0, "num_pos": 0}

    pbar = tqdm(dataloader, desc="Validate", leave=False)
    for images, targets in pbar:
        images = images.to(device)
        targets = [t.to(device) for t in targets]
        cls_weight = class_weight(targets, model.nc_external, device)

        pred = model(images)

        if torch.isnan(pred).any() or torch.isinf(pred).any():
            print("⚠️  Warning: pred has NaN/Inf during validation — skipping batch")
            continue

        loss, logs = loss_fn(
            pred, targets,
            pos_weight=cls_weight,
            num_classes=model.nc_external,
            imgsz=imgsz,
            device=device,
            weight_box=7.5,
            weight_obj=10.0,
            weight_cls=0.5
        )

        total_loss += loss.item()
        for k in loss_logs:
            loss_logs[k] += logs[k]

    avg_loss = total_loss / len(dataloader) if len(dataloader) > 0 else 0.0
    for k in loss_logs:
        loss_logs[k] /= len(dataloader) if len(dataloader) > 0 else 1

    return avg_loss, loss_logs

In [18]:
from detection_loss import detection_loss
from yolo_model import create_yolo_model
from data_proc import YOLODataset
from torch.utils.data import Dataset, DataLoader
import math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = "cpu"
print("Device:", device)

version = 7
imgsz = 640
batch_size = 16
num_epochs = 100
num_classes = 5  # ball, coach, goalkeeper, player, referee

model = create_yolo_model(num_classes, imgsz=640)
model.to(device)

train_dataset = YOLODataset(GENERAL_PATH / "images/train", GENERAL_PATH / "labels/train", GENERAL_PATH / "train_patches",imgsz=imgsz)
val_dataset = YOLODataset(GENERAL_PATH / "images/val", GENERAL_PATH / "labels/val", GENERAL_PATH / "train_patches", imgsz=imgsz, augment=False)

def collate_fn(batch):
    images, targets = zip(*batch)
    images = torch.stack(images, 0)
    return images, list(targets)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                            num_workers=0, pin_memory=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                        num_workers=0, pin_memory=True, collate_fn=collate_fn)

# 0.01 обычно для бОльших батчей
optimizer = optim.SGD(model.parameters(), lr=0.005, momentum=0.937, weight_decay=0.0005, nesterov=True)
scaler = torch.amp.GradScaler()

# разогрев для первых эпох, чтобы градиенты не улетали, далее плавное сниэение через cos^2
def lr_lambda(epoch):
    if epoch < 3:
        return (epoch + 1) / 3
    return 0.5 * (1 + math.cos(math.pi * (epoch - 3) / (num_epochs - 3)))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

min_val_loss = 0
for epoch in range(num_epochs):
    train_loss, train_logs = train_one_epoch(model, train_loader, optimizer, scaler, detection_loss, device, epoch, imgsz)
    val_loss, val_logs = validate_one_epoch(model, val_loader, detection_loss, device, imgsz)
    scheduler.step()

    tqdm.write(
        f"Epoch {epoch} | "
        f"Train: loss={train_loss:.4f} (box={train_logs['loss_box']:.4f}, obj={train_logs['loss_obj']:.4f}, cls={train_logs['loss_cls']:.4f}) | "
        f"Val: loss={val_loss:.4f} (box={val_logs['loss_box']:.4f}, obj={val_logs['loss_obj']:.4f}, cls={val_logs['loss_cls']:.4f}) | "
        f"pos: {train_logs['num_pos']}/{val_logs['num_pos']}"
    )

    if val_loss < min_val_loss or epoch == 0:
        min_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        },  WEIGHTS_PATH + f"yolo_best_val_v{version}.pt")

model.eval()
model.apply(lambda m: hasattr(m, 'reparameterize') and m.reparameterize())
torch.save(model.state_dict(), WEIGHTS_PATH + f"yolo_final_rep_v{version}.pt")
print("✅ Model reparameterized and saved for inference!")

Device: cuda
Собрано 2062 редких патчей: {0: 591, 1: 164, 2: 219, 4: 1088}


Epoch 0: 100%|██████████| 40/40 [00:38<00:00,  1.05it/s, loss=3.7982, obj=0.0096, cls=0.5542, box=0.4566, pos=2931]


Epoch 0 | Train: loss=4.3581 (box=0.4998, obj=0.0229, cls=0.7614) | Val: loss=4.1103 (box=0.4865, obj=0.0117, cls=0.6901) | pos: 2876.65/2716.4


Epoch 1: 100%|██████████| 40/40 [00:38<00:00,  1.05it/s, loss=3.2879, obj=0.0031, cls=0.2846, box=0.4153, pos=2888]


Epoch 1 | Train: loss=3.4450 (box=0.4317, obj=0.0047, cls=0.3198) | Val: loss=3.2348 (box=0.4004, obj=0.0032, cls=0.4008) | pos: 2920.0/2729.1


...

Epoch 97 | Train: loss=1.6746 (box=0.2205, obj=0.0009, cls=0.0234) | Val: loss=1.6933 (box=0.2183, obj=0.0008, cls=0.0966) | pos: 3251.975/2818.9


Epoch 98: 100%|██████████| 40/40 [00:52<00:00,  1.30s/it, loss=1.6440, obj=0.0010, cls=0.0137, box=0.2170, pos=3500]


Epoch 98 | Train: loss=1.6779 (box=0.2208, obj=0.0009, cls=0.0249) | Val: loss=1.6940 (box=0.2182, obj=0.0008, cls=0.0988) | pos: 3261.075/2818.8


Epoch 99: 100%|██████████| 40/40 [00:48<00:00,  1.20s/it, loss=1.5748, obj=0.0009, cls=0.0121, box=0.2079, pos=3398]


Epoch 99 | Train: loss=1.6861 (box=0.2220, obj=0.0009, cls=0.0241) | Val: loss=1.6943 (box=0.2183, obj=0.0008, cls=0.0979) | pos: 3272.65/2818.9
✅ Model reparameterized and saved for inference!


In [24]:
import torch
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from torchvision import transforms
from torchvision.ops import nms, box_convert


def predict_and_visualize_v2(
    model,
    image,
    class_names,
    device='cuda',
    imgsz=640,
    conf_thres=0.5,
    iou_thres=0.45,
    max_dets=300
):
    """
    Predict and visualize with simple resize (no letterbox), PIL-based loading.
    
    Args:
        model: trained model → [B, N, 9]
        image: str (path) or PIL.Image
        class_names: list or dict, e.g. ['ball', 'player', ...]
        device: 'cuda' or 'cpu'
        imgsz: resize to (imgsz, imgsz) — no letterbox
        conf_thres: obj × cls confidence threshold
        iou_thres: NMS IoU threshold
        max_dets: max detections after NMS
    
    Returns:
        PIL.Image with boxes and labels
    """
    model.eval()
    
    pil_img_orig = Image.open(image).convert("RGB")
    
    transform = transforms.Compose([
        transforms.Resize((imgsz, imgsz)),
        transforms.ToTensor(),
    ])
    x = transform(pil_img_orig).unsqueeze(0).to(device)  # [1, 3, 640, 640]

    with torch.no_grad(), torch.amp.autocast(device):
        pred = model(x)  # [1, N, 9]

    pred = pred[0]  # [N, 9]
    box_cxcywh = pred[:, :4]          # [N, 4]
    obj_logit = pred[:, 4]            # [N]
    cls_logits = pred[:, 5:]          # [N, C]
    
    obj_conf = obj_logit.sigmoid()
    cls_conf = cls_logits.sigmoid()
    class_conf, class_id = cls_conf.max(dim=1)
    conf = obj_conf * class_conf

    keep = conf > conf_thres
    if keep.sum() == 0:
        return pil_img_orig.copy()

    box_cxcywh = box_cxcywh[keep]
    conf = conf[keep]
    class_id = class_id[keep]

    box_xyxy = box_convert(box_cxcywh, in_fmt='cxcywh', out_fmt='xyxy')

    keep_nms = []
    for cls in torch.unique(class_id):
        cls_mask = class_id == cls
        cls_boxes = box_xyxy[cls_mask]
        cls_conf = conf[cls_mask]
        cls_keep = nms(cls_boxes, cls_conf, iou_threshold=iou_thres)
        keep_nms.append(torch.where(cls_mask)[0][cls_keep])
    if keep_nms:
        keep_nms = torch.cat(keep_nms)
        keep_nms = keep_nms[conf[keep_nms].argsort(descending=True)[:max_dets]]
    else:
        keep_nms = torch.tensor([], dtype=torch.long)
    if len(keep_nms) == 0:
        return pil_img_orig.copy()
    box_xyxy = box_xyxy[keep_nms]
    conf = conf[keep_nms]
    class_id = class_id[keep_nms]


    orig_w, orig_h = pil_img_orig.size
    box_xyxy[:, [0, 2]] *= orig_w / imgsz
    box_xyxy[:, [1, 3]] *= orig_h / imgsz

    box_xyxy = box_xyxy.round().int()

    result_img = pil_img_orig.copy()
    draw = ImageDraw.Draw(result_img)

    try:
        font = ImageFont.truetype("DejaVuSans.ttf", size=16)
    except:
        font = ImageFont.load_default()

    colors = [
        (255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0), (0, 255, 255),
        (255, 0, 255), (128, 0, 0), (0, 128, 0), (0, 0, 128), (128, 128, 0)
    ]

    for i in range(len(box_xyxy)):
        x1, y1, x2, y2 = box_xyxy[i].tolist()
        cls = int(class_id[i].item())
        prob = conf[i].item()
        
        label = f"{class_names[cls]} {prob:.2f}"
        color = colors[cls % len(colors)]
        
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)  # ширина 3 → лучше видно
        text_x = x1
        text_y = max(y1 - 25, 0)
        
        try:
            font = ImageFont.truetype("Arial.ttf", 40)
        except:
            try:
                font = ImageFont.truetype("DejaVuSans-Bold.ttf", 40)
            except:
                font = ImageFont.load_default(20)
        
        # Белый жирный текст
        draw.text((text_x, text_y), label, fill=(255, 255, 255), font=font, stroke_width=1, stroke_fill=(0, 0, 0))


    return result_img

In [28]:
from yolo_model import create_yolo_model
import torch
device = 'cuda'

model = create_yolo_model().to(device).eval()
model.apply(lambda m: hasattr(m, 'reparameterize') and m.reparameterize())
class_names = ['ball', 'coach', 'goalkeeper', 'player', 'referee']  # или dict
model.load_state_dict(torch.load(WEIGHTS_PATH / 'yolo_final_rep_v7.pt'))

# Тест на изображении
img_path = TEST_PATH / "frame_22959_png.rf.328de5f62c69429926b33e5d1b1761a2.jpg"
result_img = predict_and_visualize_v2(
    model, img_path, class_names,
    device='cuda',
    imgsz=640,
    conf_thres=0.4,
    iou_thres=0.1
)

# Показать
result_img.show()

In [2]:
from yolo_model import create_yolo_model
device = 'cuda'
model = create_yolo_model().to(device).eval()
model.apply(lambda m: hasattr(m, 'reparameterize') and m.reparameterize())
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Количество параметров модели:", num_params)

Количество параметров модели: 14920584
